In [78]:
# Loading the necessary imports
import dynamiqs as dq
import jax.numpy as jnp
from pathlib import Path
from scipy.optimize import least_squares
import json
import matplotlib.pyplot as plt


In [79]:
# Building the dictionary of parameters
params = {
    'Delta_a': -30.0,
    'Delta_b': -30.0,
    'Delta_c': 290.0,

    'Xaa': 0.0014,
    'Xbb': 0.0373,
    'Xcc': 164.0,

    'Xab': 0.0072,
    'Xac': 0.671,
    'Xbc': 3.5,

    'kappa_a': 0.0042,
    'kappa_b': 0.984,
    'kappa_c': 0.0207,

    'Ea': 5.6,
    'Eb': 5.0,
    'Ec': 215.0/2,
}

In [80]:
def exact_iterative_steady_state(params, tol=1e-3, max_iter=1500, mixing=0.05):
    # Unpack parameters
    Ea, Eb, Ec = params['Ea'], params['Eb'], params['Ec']
    Da, Db, Dc = params['Delta_a'], params['Delta_b'], params['Delta_c']
    ka, kb, kc = params['kappa_a']/(2*jnp.pi), params['kappa_b']/(2*jnp.pi), params['kappa_c']/(2*jnp.pi)
    
    # Unpack ALL Kerr terms
    Xaa, Xbb, Xcc = params['Xaa'], params['Xbb'], params['Xcc']
    Xab, Xac, Xbc = params['Xab'], params['Xac'], params['Xbc']

    # 1. Start with the linear guess (no Kerr shifts)
    na = jnp.abs(Ea / (ka/2 + 1j*Da))**2
    nb = jnp.abs(Eb / (kb/2 + 1j*Db))**2
    nc = jnp.abs(Ec / (kc/2 + 1j*Dc))**2

    # 2. The Self-Consistent Loop
    for i in range(max_iter):
        # Step A: Calculate the new effective detunings using CURRENT populations
        # NOW FULLY INCLUDING ALL SELF AND CROSS KERR SHIFTS
        Da_eff = Da - Xaa * na - Xab * nb - Xac * nc
        Db_eff = Db - Xbb * nb - Xab * na - Xbc * nc
        Dc_eff = Dc - Xcc * nc - Xac * na - Xbc * nb

        # Step B: Calculate what the populations SHOULD be
        na_calc = jnp.abs(Ea / (ka/2 + 1j*Da_eff))**2
        nb_calc = jnp.abs(Eb / (kb/2 + 1j*Db_eff))**2
        nc_calc = jnp.abs(Ec / (kc/2 + 1j*Dc_eff))**2

        # Step C: Check the largest error/change between current and calculated
        error = jnp.max(jnp.array([
            jnp.abs(na_calc - na),
            jnp.abs(nb_calc - nb),
            jnp.abs(nc_calc - nc)
        ]))

        # If the numbers stopped changing, we found the exact physical root!
        if error < tol:
            print(f"Success! Converged exactly in {i} iterations.")
            break

        # Step D: Apply Damping/Mixing to prevent infinite bouncing
        # We step 5% toward the new value, and keep 95% of the old value
        na = (1 - mixing) * na + mixing * na_calc
        nb = (1 - mixing) * nb + mixing * nb_calc
        nc = (1 - mixing) * nc + mixing * nc_calc
    else:
        print("Warning: Did not perfectly converge. Try lowering the mixing parameter.")

    # 3. Final Complex Amplitude Calculation (Using all Kerr terms)
    Da_eff = Da - Xaa * na - Xab * nb - Xac * nc
    Db_eff = Db - Xbb * nb - Xab * na - Xbc * nc
    Dc_eff = Dc - Xcc * nc - Xac * na - Xbc * nb

    alpha = Ea / (ka/2 + 1j * Da_eff)
    beta  = Eb / (kb/2 + 1j * Db_eff)
    gamma = Ec / (kc/2 + 1j * Dc_eff)

    # Print the perfectly consistent results
    print("=== True Self-Consistent Steady State ===")
    print("Photon Numbers (n):")
    print(f"  Storage (na)  = {na:.7f}")
    print(f"  Readout (nb)  = {nb:.7f}")
    print(f"  Transmon (nc) = {nc:.7f}")
    
    print("\nSquared Amplitudes (|Amplitude|^2) - MUST MATCH ABOVE:")
    print(f"  Storage (|α|^2)  = {jnp.abs(alpha)**2:.7f}")
    print(f"  Readout (|β|^2)  = {jnp.abs(beta)**2:.7f}")
    print(f"  Transmon (|γ|^2) = {jnp.abs(gamma)**2:.7f}")
    
    print("\nComplex Amplitudes (Real + Imag):")
    print(f"  Storage (α)  = {alpha.real:>9.6f} + {alpha.imag:>9.6f}j")
    print(f"  Readout (β)  = {beta.real:>9.6f} + {beta.imag:>9.6f}j")
    print(f"  Transmon (γ) = {gamma.real:>9.6f} + {gamma.imag:>9.6f}j")
    print("=========================================")

    return jnp.array([na, nb, nc]), jnp.array([alpha, beta, gamma])


In [81]:
nums, amps = exact_iterative_steady_state(params)
print(nums)

Success! Converged exactly in 79 iterations.
=== True Self-Consistent Steady State ===
Photon Numbers (n):
  Storage (na)  = 0.0345935
  Readout (nb)  = 0.0267580
  Transmon (nc) = 0.1666529

Squared Amplitudes (|Amplitude|^2) - MUST MATCH ABOVE:
  Storage (|α|^2)  = 0.0345856
  Readout (|β|^2)  = 0.0267260
  Transmon (|γ|^2) = 0.1676433

Complex Amplitudes (Real + Imag):
  Storage (α)  =  0.000002 +  0.185972j
  Readout (β)  =  0.000419 +  0.163480j
  Transmon (γ) =  0.000003 + -0.409443j
[0.03459351 0.02675795 0.16665293]


In [82]:
# To Compute the starc shifted detunings :
AC_Delta_a = params['Delta_a'] - 2*params['Xaa'] * nums[0] - params['Xab'] * nums[1] - params['Xac'] * nums[2]
AC_Delta_b = params['Delta_b'] - 2*params['Xbb'] * nums[1] - params['Xab'] * nums[0] - params['Xbc'] * nums[2]
AC_Delta_c = params['Delta_c'] - 2*params['Xcc'] * nums[2] - params['Xac'] * nums[0] - params['Xbc'] * nums[1]

print(f"AC Stark Shifted Detunings: {AC_Delta_a:.3f}, {AC_Delta_b:.3f}, {AC_Delta_c:.3f}")

AC Stark Shifted Detunings: -30.112, -30.586, 235.221


In [83]:
# 2. Define the photon number states (0 through 9)
N_levels = 10
n = jnp.arange(N_levels)

# 3. Calculate the energies
# Formula: E(n) = n*Δ_tilde - (X/2)*n*(n-1)
E_a = n * AC_Delta_a - (params['Xaa'] / 2.0) * n * (n - 1)
E_b = n * AC_Delta_b - (params['Xbb'] / 2.0) * n * (n - 1)
E_c = n * AC_Delta_c - (params['Xcc'] / 2.0) * n * (n - 1)

# 4. Print the formatted results
print("=== First 10 Energy Levels in the Displaced RWA Frame (MHz) ===\n")
print(f"{'|n⟩':<6} | {'Storage (a)':<15} | {'Readout (b)':<15} | {'Transmon (c)':<15}")
print("-" * 60)

for i in range(N_levels):
    print(f"|{i}⟩    | {E_a[i]:>10.4f}      | {E_b[i]:>10.4f}      | {E_c[i]:>10.4f}")

=== First 10 Energy Levels in the Displaced RWA Frame (MHz) ===

|n⟩    | Storage (a)     | Readout (b)     | Transmon (c)   
------------------------------------------------------------
|0⟩    |     0.0000      |     0.0000      |     0.0000
|1⟩    |   -30.1121      |   -30.5855      |   235.2210
|2⟩    |   -60.2256      |   -61.2084      |   306.4419
|3⟩    |   -90.3405      |   -91.8685      |   213.6629
|4⟩    |  -120.4569      |  -122.5659      |   -43.1161
|5⟩    |  -150.5746      |  -153.3007      |  -463.8951
|6⟩    |  -180.6937      |  -184.0727      | -1048.6742
|7⟩    |  -210.8142      |  -214.8820      | -1797.4532
|8⟩    |  -240.9361      |  -245.7287      | -2710.2324
|9⟩    |  -271.0594      |  -276.6126      | -3787.0112
